# 04c Fold Analysis

Per-fold AUROC/F1 breakdown, fold consistency analysis, and deployment checkpoint selection.

- Instability rule: AUROC std dev > 0.02
- Primary selection metric: `val_auroc` when available, otherwise fallback to `test_auroc`
- Tie-break metric: `val_f1` when available, otherwise `test_f1`


In [ ]:
from __future__ import annotations

import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

parquet_candidates = [
    Path(r"C:/Users/acer/Downloads/results (1).parquet"),
    Path(r"C:/Users/acer/Downloads/results.parquet"),
    RESULTS_DIR / "results.parquet",
    PROJECT_ROOT.parent / "results.parquet",
    Path(r"C:/Users/acer/OneDrive/سطح المكتب/New folder (3)/results.parquet"),
]

parquet_path = next((p for p in parquet_candidates if p.exists()), None)
if parquet_path is None:
    raise FileNotFoundError(f"Could not find results.parquet in any expected location: {parquet_candidates}")

df = pd.read_parquet(parquet_path)
if "fold" not in df.columns:
    df["fold"] = df["run_name"].astype(str).str.extract(r"fold(\\d+)").astype(float)

if df["fold"].isna().any():
    raise ValueError("Failed to infer fold values from run_name for one or more rows.")

df["fold"] = df["fold"].astype(int)

score_col = "val_auroc" if "val_auroc" in df.columns else "test_auroc"
f1_col = "val_f1" if "val_f1" in df.columns else "test_f1"

print(f"Using parquet: {parquet_path}")
print(f"Selection metric: {score_col}")
print(f"Tie-break metric: {f1_col}")
print(f"Rows: {len(df)}")
display(df.head())


In [ ]:
per_fold = (
    df[["category", "fold", "run_id", "run_name", score_col, f1_col]]
    .rename(columns={score_col: "auroc", f1_col: "f1"})
    .sort_values(["category", "fold"]) 
    .reset_index(drop=True)
)

per_fold.to_csv(RESULTS_DIR / "fold_breakdown.csv", index=False)
display(per_fold)


In [ ]:
consistency = (
    per_fold.groupby("category", as_index=False)
    .agg(
        auroc_mean=("auroc", "mean"),
        auroc_std=("auroc", "std"),
        f1_mean=("f1", "mean"),
        n_folds=("fold", "nunique"),
    )
    .sort_values("category")
)
consistency["unstable"] = consistency["auroc_std"] > 0.02

unstable_categories = consistency.loc[consistency["unstable"], "category"].tolist()
print("Unstable categories (std > 0.02):", unstable_categories if unstable_categories else "None")
display(consistency)


In [ ]:
best_rows = (
    per_fold.sort_values(["category", "auroc", "f1", "fold"], ascending=[True, False, False, True])
    .groupby("category", as_index=False)
    .head(1)
    .sort_values("category")
    .reset_index(drop=True)
)

best_rows["checkpoint_filename"] = best_rows.apply(
    lambda r: f"{r['category']}_r100_f{int(r['fold'])}_best.ckpt", axis=1
)
best_rows["checkpoint_path_hint"] = best_rows["checkpoint_filename"].map(
    lambda x: f"Kaggle Dataset checkpoint package / {x}"
)

best_payload = {
    "_metadata": {
        "source_parquet": str(parquet_path),
        "selection_metric_used": score_col,
        "tie_break_metric_used": f1_col,
        "note": "If val metrics are absent in parquet, test metrics are used as fallback."
    }
}

for _, row in best_rows.iterrows():
    best_payload[row["category"]] = {
        "selected_fold": int(row["fold"]),
        "selection_metric": score_col,
        "tie_break_metric": f1_col,
        "run_id": row["run_id"],
        "run_name": row["run_name"],
        score_col: float(row["auroc"]),
        f1_col: float(row["f1"]),
        "checkpoint_filename": row["checkpoint_filename"],
        "checkpoint_path_hint": row["checkpoint_path_hint"],
    }

with open(RESULTS_DIR / "best_checkpoints.json", "w", encoding="utf-8") as f:
    json.dump(best_payload, f, indent=2, ensure_ascii=False)

display(best_rows)
print(f"Wrote: {RESULTS_DIR / 'best_checkpoints.json'}")


In [ ]:
plt.figure(figsize=(12, 6))
ax = sns.barplot(data=per_fold, x="category", y="auroc", hue="fold", palette="viridis")
ax.set_title("Per-fold AUROC by category")
ax.set_xlabel("Category")
ax.set_ylabel("AUROC")
ax.legend(title="Fold")
plt.xticks(rotation=25)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "fold_auroc_bars.png", dpi=200)
plt.show()


In [ ]:
heat_df = per_fold.pivot(index="category", columns="fold", values="auroc").sort_index()
plt.figure(figsize=(8, 5))
ax = sns.heatmap(heat_df, annot=True, fmt=".4f", cmap="mako", cbar_kws={"label": "AUROC"})
ax.set_title("Fold consistency heatmap (AUROC)")
ax.set_xlabel("Fold")
ax.set_ylabel("Category")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "fold_consistency_heatmap.png", dpi=200)
plt.show()
